In [11]:
import os
import data
import random
import sklearn
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.decomposition import NMF
from sklearn.utils import shuffle, check_random_state

import config
from utils import apply_symmetric_noise

random.seed(config.SEED)
np.random.seed(config.SEED)
np.random.default_rng(config.SEED)
check_random_state(config.SEED)

# import warnings

# warnings.filterwarnings("ignore")

RandomState(MT19937) at 0x133A5D240

In [3]:
array_utk = shuffle(data.load_utk())

array_utk_noisy = apply_symmetric_noise(array_utk, prob=0.15)[:1500]

In [6]:
def cosine_distance(u: np.ndarray, v: np.ndarray) -> float:
    u = np.asarray(u, dtype=np.float64).ravel()
    v = np.asarray(v, dtype=np.float64).ravel()
    uu = np.dot(u, u); vv = np.dot(v, v)
    if uu <= 0.0 or vv <= 0.0:
        return 1.0
    return 1.0 - (np.dot(u, v) / np.sqrt(uu * vv))

In [16]:
def nmf_reconstruct_frob(X_noisy: np.ndarray, rank: int,
                         max_iter: int = 20000, tol: float = 1e-10) -> np.ndarray:
    model = NMF(n_components=rank, init="random", max_iter=max_iter, tol=tol)
    W = model.fit_transform(X_noisy)
    H = model.components_ 
    return W @ H  

In [17]:
def calc_deltas(X_clean: np.ndarray, X_noisy: np.ndarray, X_hat_noisy: np.ndarray) -> np.ndarray:
    M = X_clean.shape[0]
    deltas = np.empty(M, dtype=np.float64)
    for i in range(M):
        d1 = cosine_distance(X_clean[i], X_noisy[i])
        d2 = cosine_distance(X_clean[i], X_hat_noisy[i])
        deltas[i] = d1 - d2
    return deltas

In [18]:
array_utk_recon = nmf_reconstruct_frob(array_utk_noisy, 32)

/Users/edvard/Desktop/NNMF/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/edvard/Desktop/NNMF/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/edvard/Desktop/NNMF/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/var/folders/x5/8yrnb6h17yq2klc6rfv8x3xr0000gn/T/ipykernel_37788/3512265694.py:6: RuntimeWarning: divide by zero encountered in matmul
  return W @ H
/var/folders/x5/8yrnb6h17yq2klc6rfv8x3xr0000gn/T/ipykernel_37788/3512265694.py:6: RuntimeWarning: overflow encountered in matmul
  return W @ H
/var/folders/x5/8yrnb6h17yq2klc6rfv8x3xr0000gn/T/ipykernel_37788/3512265694.py:6: RuntimeWarning: invalid value encountered in matmul
  return W @ H
